In [ ]:
"""
feedback_analyzer.py

Metrics:
  1. Peak Extension (Elbow Angle)        — Jab, Cross only
  2. Guard Hand Drop (DTW frame-wise)    — All punches
  3. Punch Duration                      — All punches
  4. Retraction Speed                    — Jab, Cross only
  5. Shoulder Rotation (Total Arc)       — All punches
  6. Hip Rotation (Total Arc)            — All punches
  7. Arm Compactness (Wrist-Shoulder)    — Lead Hook, Rear Hook, Lead Uppercut, Rear Uppercut
  8. Hook Elbow Height                   — Lead Hook, Rear Hook only
  9. Foot Stance Width (Pre-punch)       — All punches
"""

import os
import sys
import cv2
import numpy as np
import mediapipe as mp
from dtaidistance import dtw_ndim
from dtaidistance import dtw as dtw1d


# ─────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────

def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle   = abs((180.0 * radians) / np.pi)
    return 360.0 - angle if angle > 180.0 else angle

def extract_angles(frame):
    def get_xy(idx): return [frame[idx*2], frame[idx*2+1]]
    return [
        calculate_angle(get_xy(0), get_xy(2), get_xy(4)),
        calculate_angle(get_xy(1), get_xy(3), get_xy(5)),
        calculate_angle(get_xy(6), get_xy(0), get_xy(2)),
        calculate_angle(get_xy(7), get_xy(1), get_xy(3)),
        calculate_angle(get_xy(0), get_xy(6), get_xy(8)),
        calculate_angle(get_xy(1), get_xy(7), get_xy(9)),
    ]

def extract_velocities(curr, prev):
    v = []
    for j in range(6):
        v.extend([curr[j*2]-prev[j*2], curr[j*2+1]-prev[j*2+1]])
    return v

def extract_wrist_speeds(velocities):
    lw = np.sqrt(velocities[8]**2  + velocities[9]**2)
    rw = np.sqrt(velocities[10]**2 + velocities[11]**2)
    return [lw, rw]

def wrist_to_shoulder_distance(frame):
    return [np.linalg.norm(np.array(frame[8:10])  - np.array(frame[0:2])),
            np.linalg.norm(np.array(frame[10:12]) - np.array(frame[2:4]))]

def lead_rear_asymmetry(frame):
    return [frame[8] - frame[10]]

def build_feature_vector(curr, prev):
    velocities = extract_velocities(curr, prev)
    return np.concatenate([
        curr, velocities, extract_angles(curr),
        extract_wrist_speeds(velocities),
        wrist_to_shoulder_distance(curr),
        lead_rear_asymmetry(curr),
    ])

def process_punch_sequence(punch):
    if len(punch) == 0:
        return []
    features = []
    zero_vel   = [0.0]*12
    zero_speed = [0.0, 0.0]
    asy        = [0]
    features.append(np.concatenate([
        punch[0], zero_vel, extract_angles(punch[0]),
        zero_speed, wrist_to_shoulder_distance(punch[0]), asy
    ]))
    for i in range(1, len(punch)):
        features.append(build_feature_vector(punch[i], punch[i-1]))
    return features


# ─────────────────────────────────────────────
# NORMALIZATION
# ─────────────────────────────────────────────

LANDMARK_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]

def translation_normalization(frame):
    hip_center = np.array([(frame[12]+frame[14])/2, (frame[13]+frame[15])/2])
    out = []
    for i in range(0, len(frame), 2):
        out.extend((np.array([frame[i], frame[i+1]]) - hip_center).tolist())
    return out

def scale_normalization(frame):
    sc = np.array([(frame[0]+frame[2])/2, (frame[1]+frame[3])/2])
    hc = np.array([(frame[12]+frame[14])/2, (frame[13]+frame[15])/2])
    torso = max(np.linalg.norm(sc - hc), 1e-6)
    return [v / torso for v in frame]

def extract_video_sequence(video_path):
    mp_pose = mp.solutions.pose
    pose    = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        sys.exit(f"[ERROR] Cannot open video: {video_path}")

    fps        = cap.get(cv2.CAP_PROP_FPS)
    width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_size = (width, height)

    raw_frames   = []
    raw_lm_list  = []
    norm_coords  = []
    ankle_data   = []   

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        results = pose.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark
            raw_coords_frame = []
            for i in LANDMARK_INDICES:
                raw_coords_frame.extend([lm[i].x, lm[i].y])
            norm = scale_normalization(translation_normalization(raw_coords_frame))

            left_ankle  = lm[27]
            right_ankle = lm[28]
            ankle_data.append((left_ankle.x, left_ankle.y, right_ankle.x, right_ankle.y))

            raw_frames.append(frame.copy())
            raw_lm_list.append([lm[i] for i in LANDMARK_INDICES])
            norm_coords.append(norm)

    cap.release()
    pose.close()

    feature_seq = process_punch_sequence(norm_coords)
    return feature_seq, raw_lm_list, fps, frame_size, raw_frames, ankle_data


# ─────────────────────────────────────────────
# FEATURE INDEX CONSTANTS
# ─────────────────────────────────────────────

# Normalized coordinate indices (0-31)
LW_X, LW_Y = 8,  9    # Left wrist
RW_X, RW_Y = 10, 11   # Right wrist
LS_X, LS_Y = 0,  1    # Left shoulder
RS_X, RS_Y = 2,  3    # Right shoulder
LE_X, LE_Y = 4,  5    # Left elbow
RE_X, RE_Y = 6,  7    # Right elbow
LH_X, LH_Y = 12, 13   # Left hip
RH_X, RH_Y = 14, 15   # Right hip

# Angles (44-49)
L_ELBOW_ANGLE = 44
R_ELBOW_ANGLE = 45

# Wrist speeds (50-51)
LW_SPEED = 50
RW_SPEED = 51

# Wrist-shoulder distances (52-53)
LW_SHLDR_DIST = 52
RW_SHLDR_DIST = 53

# Punch type groupings
STRAIGHT_PUNCHES    = {"jab", "cross"}
HOOK_PUNCHES        = {"lead_hook", "rear_hook"}
UPPERCUT_PUNCHES    = {"lead_uppercut", "rear_uppercut"}
CURVED_PUNCHES      = HOOK_PUNCHES | UPPERCUT_PUNCHES   # hooks + uppercuts

THRESHOLDS = {
    # Metric 1 — Peak Extension (straight punches only)
    "elbow_diff":              8.0,   # degrees — ref minus user peak elbow
    "jab_min_elbow":           165.0, # minimum acceptable peak elbow for jab
    "cross_min_elbow":         165.0, # minimum acceptable peak elbow for cross

    # Metric 2 — Guard Hand
    "guard_avg_diff":          0.06,
    "guard_max_diff":          0.10,

    # Metric 3 — Duration
    "duration_ratio_slow":     1.25,
    "duration_ratio_fast":     0.75,

    # Metric 4 — Retraction (straight punches only)
    "retraction_ratio":        0.80,

    # Metric 5 — Shoulder Rotation
    "shoulder_arc_ratio":      0.80,
    "shoulder_arc_min_signal": 0.05,

    # Metric 6 — Hip Rotation
    "hip_arc_ratio":           0.80,
    "hip_arc_min_signal":      0.04,

    # Metric 7 — Arm Compactness (curved punches only)
    # Delivery-phase, DTW-aligned wrist-to-shoulder distance profile comparison.
    "compactness_window":            10,    # frames each side of peak wrist speed
    "compactness_mae_threshold":     0.05,  # normalized units — below this, profiles are "close enough"
    "compactness_signed_threshold":  0.03,  # normalized units — minimum consistent bias to call a direction

    # Metric 8 — Hook Elbow Height (hooks only)
    "elbow_height_diff":       0.08,  # normalized units — user elbow lower than ref
    "elbow_height_high_diff":  0.10,  # normalized units — user elbow much higher than ref

    # Metric 9 — Foot Stance Width
    "stance_ratio_wide":       1.30,  # user stance > 1.30x ref = too wide
    "stance_ratio_narrow":     0.70,  # user stance < 0.70x ref = too narrow
    "stance_frames":           10,    # number of pre-punch frames to average over
}

RETRACTION_SPAN = 8


# ─────────────────────────────────────────────
# METRIC HELPERS
# ─────────────────────────────────────────────

def get_shoulder_width(frame):
    ls = np.array([frame[LS_X], frame[LS_Y]])
    rs = np.array([frame[RS_X], frame[RS_Y]])
    return float(np.linalg.norm(rs - ls))

def get_hip_width(frame):
    lh = np.array([frame[LH_X], frame[LH_Y]])
    rh = np.array([frame[RH_X], frame[RH_Y]])
    return float(np.linalg.norm(rh - lh))

def get_wrist_shoulder_dist(frame, is_lead):
    """Returns punching wrist to punching shoulder distance."""
    if is_lead:
        wrist    = np.array([frame[LW_X], frame[LW_Y]])
        shoulder = np.array([frame[LS_X], frame[LS_Y]])
    else:
        wrist    = np.array([frame[RW_X], frame[RW_Y]])
        shoulder = np.array([frame[RS_X], frame[RS_Y]])
    return float(np.linalg.norm(wrist - shoulder))

def compute_total_arc(seq, width_fn):
    """
    Sum of absolute frame-to-frame changes in width.
    No cancellation between positive/negative movements.
    Works correctly for hooks/uppercuts where rotation reverses direction.
    """
    total = 0.0
    for i in range(1, len(seq)):
        total += abs(width_fn(seq[i]) - width_fn(seq[i-1]))
    return total

def compute_stance_width(ankle_data, n_frames=10):
    """
    Averages ankle distance over first n_frames stable pre-punch frames.
    ankle_data: list of (lax, lay, rax, ray) normalized coords.
    Returns normalized ankle-to-ankle distance.
    """
    n = min(n_frames, len(ankle_data))
    if n == 0:
        return 0.0
    widths = []
    for i in range(n):
        la = np.array([ankle_data[i][0], ankle_data[i][1]])
        ra = np.array([ankle_data[i][2], ankle_data[i][3]])
        widths.append(float(np.linalg.norm(ra - la)))
    return float(np.mean(widths))

def get_delivery_phase_window(arr, speed_idx, half_window):
    """
    Returns (start, end, peak_idx) — a window of frames centered on peak
    wrist speed (the delivery/impact moment), clipped to sequence bounds.
    Used to restrict Metric 7 to the actual punch delivery instead of the
    whole clip (stance, guard, retraction, etc. aren't relevant to
    "how open was the arm during the punch").
    """
    peak_idx = int(np.argmax(arr[:, speed_idx]))
    start = max(0, peak_idx - half_window)
    end   = min(len(arr), peak_idx + half_window + 1)
    return start, end, peak_idx


# ─────────────────────────────────────────────
# FEEDBACK ENGINE
# ─────────────────────────────────────────────

def compute_feedback(user_seq, ref_seq, path, punch_type,
                     user_ankle_data, ref_ankle_data, log_lines):
    punch_type = punch_type.lower()
    is_lead    = punch_type in {"jab", "lead_hook", "lead_uppercut"}

    elbow_idx   = L_ELBOW_ANGLE if is_lead else R_ELBOW_ANGLE
    speed_idx   = LW_SPEED      if is_lead else RW_SPEED
    guard_y_idx = RW_Y          if is_lead else LW_Y

    # Elbow and shoulder y indices for hook elbow height metric
    elbow_y_idx    = LE_Y if is_lead else RE_Y
    shoulder_y_idx = LS_Y if is_lead else RS_Y

    user_arr = np.array(user_seq)
    ref_arr  = np.array(ref_seq)

    feedback = []
    log_lines.append("=" * 60)
    log_lines.append(f"PUNCH TYPE : {punch_type.upper()}")
    log_lines.append(f"USER FRAMES: {len(user_arr)}  |  REF FRAMES: {len(ref_arr)}")
    log_lines.append("=" * 60)

    # ── METRIC 1: Peak Extension (Straight punches only) ──────────
    log_lines.append("\n[METRIC 1] Peak Extension (Elbow Angle)")
    if punch_type not in STRAIGHT_PUNCHES:
        log_lines.append(f"  SKIPPED — not applicable for {punch_type}")
    else:
        min_elbow_key = f"{punch_type}_min_elbow"
        elbow_threshold = THRESHOLDS.get(min_elbow_key, 165.0)

        user_peak_elbow = float(np.max(user_arr[:, elbow_idx]))
        ref_peak_elbow  = float(np.max(ref_arr[:,  elbow_idx]))
        elbow_diff      = ref_peak_elbow - user_peak_elbow

        log_lines.append(f"  User peak elbow : {user_peak_elbow:.1f} deg")
        log_lines.append(f"  Ref  peak elbow : {ref_peak_elbow:.1f} deg")
        log_lines.append(f"  Difference      : {elbow_diff:.1f} deg")
        log_lines.append(f"  Threshold diff  : {THRESHOLDS['elbow_diff']} deg")
        log_lines.append(f"  Min elbow needed: {elbow_threshold} deg")

        if elbow_diff > THRESHOLDS["elbow_diff"] and user_peak_elbow < elbow_threshold:
            feedback.append("Extend your elbow more during the punch")
            log_lines.append("  -> FEEDBACK FIRED")
        else:
            log_lines.append("  -> OK")

    # ── METRIC 2: Guard Hand (DTW frame-wise) 
    guard_diffs = []
    for (u_idx, r_idx) in path:
        if u_idx < len(user_arr) and r_idx < len(ref_arr):
            guard_diffs.append(
                float(user_arr[u_idx, guard_y_idx]) - float(ref_arr[r_idx, guard_y_idx])
            )

    guard_avg = float(np.mean(guard_diffs)) if guard_diffs else 0.0
    guard_max = float(np.max(guard_diffs))  if guard_diffs else 0.0

    log_lines.append("\n[METRIC 2] Guard Hand (DTW frame-wise)")
    log_lines.append(f"  Avg guard drop  : {guard_avg:.4f}")
    log_lines.append(f"  Max guard drop  : {guard_max:.4f}")
    log_lines.append(f"  Threshold avg   : {THRESHOLDS['guard_avg_diff']}")
    log_lines.append(f"  Threshold max   : {THRESHOLDS['guard_max_diff']}")

    if guard_avg > THRESHOLDS["guard_avg_diff"] or guard_max > THRESHOLDS["guard_max_diff"]:
        feedback.append("Keep your guard hand up while punching")
        log_lines.append("  -> FEEDBACK FIRED")
    else:
        log_lines.append("  -> OK")

    # ── METRIC 3: Punch Duration ───────────────────────────────────
    dur_ratio = len(user_arr) / len(ref_arr) if len(ref_arr) > 0 else 1.0

    log_lines.append("\n[METRIC 3] Punch Duration")
    log_lines.append(f"  User frames    : {len(user_arr)}")
    log_lines.append(f"  Ref  frames    : {len(ref_arr)}")
    log_lines.append(f"  Duration ratio : {dur_ratio:.3f}")
    log_lines.append(f"  Slow threshold : {THRESHOLDS['duration_ratio_slow']}")
    log_lines.append(f"  Fast threshold : {THRESHOLDS['duration_ratio_fast']}")

    if dur_ratio > THRESHOLDS["duration_ratio_slow"]:
        feedback.append("Increase your punch speed")
        log_lines.append("  -> FEEDBACK FIRED (too slow)")
    elif dur_ratio < THRESHOLDS["duration_ratio_fast"]:
        feedback.append("Slow down slightly — focus on full extension")
        log_lines.append("  -> FEEDBACK FIRED (too fast)")
    else:
        log_lines.append("  -> OK")

    # ── METRIC 4: Retraction Speed (Straight punches only) ────────
    log_lines.append("\n[METRIC 4] Retraction Speed")
    if punch_type not in STRAIGHT_PUNCHES:
        log_lines.append(f"  SKIPPED — not applicable for {punch_type}")
    else:
        u_peak_idx = int(np.argmax(user_arr[:, elbow_idx]))
        r_peak_idx = int(np.argmax(ref_arr[:,  elbow_idx]))

        u_post = user_arr[u_peak_idx:u_peak_idx+RETRACTION_SPAN, speed_idx]
        r_post = ref_arr[r_peak_idx:r_peak_idx+RETRACTION_SPAN,  speed_idx]

        if len(u_post) > 0 and len(r_post) > 0:
            u_ret = float(np.mean(u_post))
            r_ret = float(np.mean(r_post))
            ret_ratio = u_ret / r_ret if r_ret > 1e-6 else 1.0

            log_lines.append(f"  User peak frame       : {u_peak_idx}")
            log_lines.append(f"  Ref  peak frame       : {r_peak_idx}")
            log_lines.append(f"  User retraction speed : {u_ret:.4f}")
            log_lines.append(f"  Ref  retraction speed : {r_ret:.4f}")
            log_lines.append(f"  Ratio (user/ref)      : {ret_ratio:.3f}")
            log_lines.append(f"  Threshold             : {THRESHOLDS['retraction_ratio']}")

            if r_ret > 1e-6 and ret_ratio < THRESHOLDS["retraction_ratio"]:
                feedback.append("Retract your hand faster after impact")
                log_lines.append("  -> FEEDBACK FIRED")
            else:
                log_lines.append("  -> OK")
        else:
            log_lines.append("  Not enough post-peak frames to measure")

    # ── METRIC 5: Shoulder Rotation (Total Arc) ───────────────────
    user_shoulder_arc  = compute_total_arc(user_arr, get_shoulder_width)
    ref_shoulder_arc   = compute_total_arc(ref_arr,  get_shoulder_width)
    shoulder_arc_ratio = user_shoulder_arc / ref_shoulder_arc if ref_shoulder_arc > 1e-6 else 1.0

    log_lines.append("\n[METRIC 5] Shoulder Rotation (Total Arc Method)")
    log_lines.append(f"  User shoulder arc    : {user_shoulder_arc:.4f}")
    log_lines.append(f"  Ref  shoulder arc    : {ref_shoulder_arc:.4f}")
    log_lines.append(f"  Arc ratio (user/ref) : {shoulder_arc_ratio:.3f}")
    log_lines.append(f"  Min ref arc signal   : {THRESHOLDS['shoulder_arc_min_signal']}")
    log_lines.append(f"  Fire threshold ratio : {THRESHOLDS['shoulder_arc_ratio']}")

    if ref_shoulder_arc < THRESHOLDS["shoulder_arc_min_signal"]:
        log_lines.append("  SKIPPED — reference shoulder arc too small")
    elif shoulder_arc_ratio < THRESHOLDS["shoulder_arc_ratio"]:
        feedback.append("Rotate your shoulders more to generate power")
        log_lines.append("  -> FEEDBACK FIRED")
    else:
        log_lines.append("  -> OK")

    # ── METRIC 6: Hip Rotation (Total Arc) ────────────────────────
    user_hip_arc  = compute_total_arc(user_arr, get_hip_width)
    ref_hip_arc   = compute_total_arc(ref_arr,  get_hip_width)
    hip_arc_ratio = user_hip_arc / ref_hip_arc if ref_hip_arc > 1e-6 else 1.0

    log_lines.append("\n[METRIC 6] Hip Rotation (Total Arc Method)")
    log_lines.append(f"  User hip arc         : {user_hip_arc:.4f}")
    log_lines.append(f"  Ref  hip arc         : {ref_hip_arc:.4f}")
    log_lines.append(f"  Arc ratio (user/ref) : {hip_arc_ratio:.3f}")
    log_lines.append(f"  Min ref arc signal   : {THRESHOLDS['hip_arc_min_signal']}")
    log_lines.append(f"  Fire threshold ratio : {THRESHOLDS['hip_arc_ratio']}")

    if ref_hip_arc < THRESHOLDS["hip_arc_min_signal"]:
        log_lines.append("  SKIPPED — reference hip arc too small")
    elif hip_arc_ratio < THRESHOLDS["hip_arc_ratio"]:
        feedback.append("Rotate your hips more to generate greater power")
        log_lines.append("  -> FEEDBACK FIRED")
    else:
        log_lines.append("  -> OK")

    # ── METRIC 7: Arm Compactness (Curved punches only) ────────────
    # Delivery-phase, DTW-aligned wrist-to-shoulder distance profile.
    # 1. Per-frame wrist-shoulder distance already exists (LW/RW_SHLDR_DIST).
    # 2. Window it around peak wrist speed (the actual delivery moment) —
    #    avoids picking up spurious values from stance/guard/retraction.
    # 3. DTW-align the two windowed 1-D profiles (independent of the
    #    whole-body path passed in, since that path was optimized for the
    #    full multi-dimensional pose signal, not this specific scalar).
    # 4. Mean absolute error = how different; mean signed error = direction.
    # 5. MAE gates whether there's a meaningful difference at all; the sign
    #    of the mean signed error (only consulted if MAE says "different")
    #    decides which direction of feedback to give.
    log_lines.append("\n[METRIC 7] Arm Compactness (Wrist-Shoulder Distance)")
    if punch_type not in CURVED_PUNCHES:
        log_lines.append(f"  SKIPPED — not applicable for {punch_type}")
    else:
        dist_idx = LW_SHLDR_DIST if is_lead else RW_SHLDR_DIST
        half_win = int(THRESHOLDS["compactness_window"])

        u_start, u_end, u_peak = get_delivery_phase_window(user_arr, speed_idx, half_win)
        r_start, r_end, r_peak = get_delivery_phase_window(ref_arr,  speed_idx, half_win)

        u_series = user_arr[u_start:u_end, dist_idx].astype(np.double)
        r_series = ref_arr[r_start:r_end,  dist_idx].astype(np.double)

        log_lines.append(f"  User peak-speed frame : {u_peak}  (window {u_start}-{u_end-1})")
        log_lines.append(f"  Ref  peak-speed frame : {r_peak}  (window {r_start}-{r_end-1})")

        if len(u_series) < 2 or len(r_series) < 2:
            log_lines.append("  SKIPPED — not enough frames in delivery-phase window")
        else:
            local_path = dtw1d.warping_path(u_series, r_series)
            errors = np.array([u_series[ui] - r_series[ri] for (ui, ri) in local_path])
            mae = float(np.mean(np.abs(errors)))
            mean_signed = float(np.mean(errors))

            log_lines.append(f"  Mean absolute error   : {mae:.4f}")
            log_lines.append(f"  Mean signed error     : {mean_signed:.4f}  (+ = user more open, - = user more compact)")
            log_lines.append(f"  MAE threshold         : {THRESHOLDS['compactness_mae_threshold']}")
            log_lines.append(f"  Signed threshold      : {THRESHOLDS['compactness_signed_threshold']}")

            if mae < THRESHOLDS["compactness_mae_threshold"]:
                log_lines.append("  -> OK (delivery-phase profiles closely matched)")
            elif mean_signed > THRESHOLDS["compactness_signed_threshold"]:
                feedback.append("Keep your punch compact — avoid letting your arm flare outward")
                log_lines.append("  -> FEEDBACK FIRED (user consistently more open than reference)")
            elif mean_signed < -THRESHOLDS["compactness_signed_threshold"]:
                feedback.append("Allow your arm to open more and extend through the punch")
                log_lines.append("  -> FEEDBACK FIRED (user consistently more compact than reference)")
            else:
                log_lines.append("  -> OK (difference not directionally consistent — likely timing noise)")

    # ── METRIC 8: Hook Elbow Height (Hooks only) ──────────────────
    # At peak wrist speed frame (impact moment), compare elbow y relative to shoulder y.
    # Elbow should be at roughly shoulder height for a correct hook.
    log_lines.append("\n[METRIC 8] Hook Elbow Height")
    if punch_type not in HOOK_PUNCHES:
        log_lines.append(f"  SKIPPED — not applicable for {punch_type}")
    else:
        # Peak wrist speed = impact moment for hooks
        u_impact_idx = int(np.argmax(user_arr[:, speed_idx]))
        r_impact_idx = int(np.argmax(ref_arr[:,  speed_idx]))

        # Elbow height relative to shoulder (negative = elbow above shoulder = good)
        u_elbow_rel = float(user_arr[u_impact_idx, elbow_y_idx]) - \
                      float(user_arr[u_impact_idx, shoulder_y_idx])
        r_elbow_rel = float(ref_arr[r_impact_idx,  elbow_y_idx]) - \
                      float(ref_arr[r_impact_idx,   shoulder_y_idx])

        height_diff = u_elbow_rel - r_elbow_rel
        # Positive = user elbow is LOWER than reference = dropped elbow
        # Negative = user elbow is HIGHER than reference

        log_lines.append(f"  User impact frame         : {u_impact_idx}")
        log_lines.append(f"  Ref  impact frame         : {r_impact_idx}")
        log_lines.append(f"  User elbow rel to shoulder: {u_elbow_rel:.4f}")
        log_lines.append(f"  Ref  elbow rel to shoulder: {r_elbow_rel:.4f}")
        log_lines.append(f"  Difference (user - ref)   : {height_diff:.4f}")
        log_lines.append(f"  Low threshold             : {THRESHOLDS['elbow_height_diff']}")
        log_lines.append(f"  High threshold            : {THRESHOLDS['elbow_height_high_diff']}")

        if height_diff > THRESHOLDS["elbow_height_diff"]:
            feedback.append("Keep your elbow higher during the hook")
            log_lines.append("  -> FEEDBACK FIRED (elbow too low)")
        elif height_diff < -THRESHOLDS["elbow_height_high_diff"]:
            feedback.append("Maintain a natural elbow position — avoid exaggerating the hook")
            log_lines.append("  -> FEEDBACK FIRED (elbow too high)")
        else:
            log_lines.append("  -> OK")

    # ── METRIC 9: Foot Stance Width (Pre-punch) ───────────────────
    # Averages ankle distance over first N frames before the punch.
    # Normalizes by hip width to remove body size variation.
    log_lines.append("\n[METRIC 9] Foot Stance Width (Pre-punch)")
    if len(user_ankle_data) == 0 or len(ref_ankle_data) == 0:
        log_lines.append("  SKIPPED — ankle data unavailable")
    else:
        n_frames = THRESHOLDS["stance_frames"]

        # Raw ankle distances
        user_raw_stance = compute_stance_width(user_ankle_data, n_frames)
        ref_raw_stance  = compute_stance_width(ref_ankle_data,  n_frames)

        # Normalize by hip width to remove body size differences
        user_hip_w = get_hip_width(user_arr[0])
        ref_hip_w  = get_hip_width(ref_arr[0])

        user_norm_stance = user_raw_stance / user_hip_w if user_hip_w > 1e-6 else user_raw_stance
        ref_norm_stance  = ref_raw_stance  / ref_hip_w  if ref_hip_w  > 1e-6 else ref_raw_stance

        stance_ratio = user_norm_stance / ref_norm_stance if ref_norm_stance > 1e-6 else 1.0

        log_lines.append(f"  User raw ankle dist      : {user_raw_stance:.4f}")
        log_lines.append(f"  Ref  raw ankle dist      : {ref_raw_stance:.4f}")
        log_lines.append(f"  User hip width           : {user_hip_w:.4f}")
        log_lines.append(f"  Ref  hip width           : {ref_hip_w:.4f}")
        log_lines.append(f"  User normalized stance   : {user_norm_stance:.4f}")
        log_lines.append(f"  Ref  normalized stance   : {ref_norm_stance:.4f}")
        log_lines.append(f"  Stance ratio (user/ref)  : {stance_ratio:.3f}")
        log_lines.append(f"  Wide threshold           : {THRESHOLDS['stance_ratio_wide']}")
        log_lines.append(f"  Narrow threshold         : {THRESHOLDS['stance_ratio_narrow']}")

        if stance_ratio > THRESHOLDS["stance_ratio_wide"]:
            feedback.append("Reduce your stance width slightly to maintain mobility")
            log_lines.append("  -> FEEDBACK FIRED (stance too wide)")
        elif stance_ratio < THRESHOLDS["stance_ratio_narrow"]:
            feedback.append("Widen your stance slightly for a stronger base")
            log_lines.append("  -> FEEDBACK FIRED (stance too narrow)")
        else:
            log_lines.append("  -> OK")

    # ── Summary ───────────────────────────────────────────────────
    log_lines.append("\n" + "=" * 60)
    if not feedback:
        feedback.append("Excellent form! Great execution.")
        log_lines.append("RESULT: No issues detected — Excellent form.")
    else:
        log_lines.append(f"RESULT: {len(feedback)} feedback point(s) generated:")
        for f in feedback:
            log_lines.append(f"  >> {f}")
    log_lines.append("=" * 60)

    return feedback


# ─────────────────────────────────────────────
# SKELETON DRAWING
# ─────────────────────────────────────────────

SKELETON_CONNECTIONS = [
    (0,1),(0,2),(2,4),(1,3),(3,5),
    (0,6),(1,7),(6,7),(6,8),(7,9),(8,10),(9,11),
]

def compute_anchor_scale(pixel_points):
    ls, rs = np.array(pixel_points[0]), np.array(pixel_points[1])
    lh, rh = np.array(pixel_points[6]), np.array(pixel_points[7])
    shoulder_mid = (ls + rs) / 2.0
    hip_mid      = (lh + rh) / 2.0
    torso = max(np.linalg.norm(shoulder_mid - hip_mid), 1e-6)
    return hip_mid, torso

def project_reference_onto_user(ref_landmarks, ref_size, user_landmarks, user_size):
    rw, rh = ref_size
    uw, uh = user_size
    ref_px  = [(lm.x * rw, lm.y * rh) for lm in ref_landmarks]
    user_px = [(lm.x * uw, lm.y * uh) for lm in user_landmarks]
    ref_anchor,  ref_scale  = compute_anchor_scale(ref_px)
    user_anchor, user_scale = compute_anchor_scale(user_px)
    ref_unit  = [(np.array(p) - ref_anchor) / ref_scale for p in ref_px]
    projected = [(u * user_scale) + user_anchor for u in ref_unit]
    return [tuple(p.astype(int)) for p in projected]

def draw_reference_skeleton(frame, points, alpha=0.82):
    overlay = frame.copy()
    for (a, b) in SKELETON_CONNECTIONS:
        if a < len(points) and b < len(points):
            cv2.line(overlay, points[a], points[b], (0, 80, 80), 10, cv2.LINE_AA)
    for pt in points:
        cv2.circle(overlay, pt, 12, (0, 60, 60), -1, cv2.LINE_AA)
    for (a, b) in SKELETON_CONNECTIONS:
        if a < len(points) and b < len(points):
            cv2.line(overlay, points[a], points[b], (0, 255, 220), 5, cv2.LINE_AA)
    for pt in points:
        cv2.circle(overlay, pt, 7, (255, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(overlay, pt, 5, (0, 255, 220), -1, cv2.LINE_AA)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)


# ─────────────────────────────────────────────
# ANIMATED FEEDBACK RENDERER
# ─────────────────────────────────────────────


def render_feedback_frames(base_frame, feedback_list, fps, vw, hold_seconds=3):
    """
    Renders the fade-in feedback animation and writes each frame directly to
    the provided cv2.VideoWriter `vw`, instead of returning a list of frames.
    """
    h, w   = base_frame.shape[:2]
    dimmed = (base_frame * 0.45).astype(np.uint8)
    fade_steps  = int(fps * 0.4)
    hold_frames = int(fps * hold_seconds)

    revealed = []
    for line in feedback_list:
        for step in range(fade_steps):
            alpha_line = step / fade_steps
            canvas = dimmed.copy()
            _draw_feedback_text(canvas, revealed + [(line, alpha_line)], w, h)
            vw.write(canvas)
        revealed.append((line, 1.0))

    final = dimmed.copy()
    _draw_feedback_text(final, revealed, w, h)
    for _ in range(hold_frames):
        vw.write(final)

def _draw_feedback_text(canvas, lines_with_alpha, w, h):
    title       = "PUNCH ANALYSIS"
    font        = cv2.FONT_HERSHEY_SIMPLEX
    title_scale = 0.9
    line_scale  = 0.7
    line_gap    = 42
    start_y     = int(h * 0.25)

    title_size = cv2.getTextSize(title, font, title_scale, 2)[0]
    title_x    = (w - title_size[0]) // 2
    cv2.putText(canvas, title, (title_x, start_y),
                font, title_scale, (255, 215, 0), 2, cv2.LINE_AA)
    cv2.line(canvas, (int(w*0.1), start_y+12),
             (int(w*0.9), start_y+12), (255, 215, 0), 1)

    for i, (text, alpha) in enumerate(lines_with_alpha):
        y         = start_y + 55 + i * line_gap
        full_text = f"  >>  {text}"
        text_size = cv2.getTextSize(full_text, font, line_scale, 2)[0]
        x_offset  = int((1 - alpha) * w * 0.3)
        x         = max(int(w * 0.08) - x_offset, 0)
        brightness = int(180 + 75 * alpha)
        color      = (brightness, brightness, brightness)
        pad        = 8
        bg_alpha   = 0.45 * alpha
        bg_overlay = canvas.copy()
        cv2.rectangle(bg_overlay,
                      (x - pad, y - text_size[1] - pad),
                      (x + text_size[0] + pad, y + pad),
                      (40, 40, 40), -1)
        cv2.addWeighted(bg_overlay, bg_alpha, canvas, 1-bg_alpha, 0, canvas)
        cv2.putText(canvas, full_text, (x, y),
                    font, line_scale, color, 2, cv2.LINE_AA)

    footer = "AI Boxing Assistant"
    fs     = cv2.getTextSize(footer, font, 0.5, 1)[0]
    cv2.putText(canvas, footer, ((w - fs[0])//2, h - 20),
                font, 0.5, (120, 120, 120), 1, cv2.LINE_AA)


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────

def main():
    # ── HARDCODED CONFIGURATION ───────────────────────────────────
    USER_VIDEO_PATH = r"E:\Person2\jab\jab-00.00.02.598-00.00.03.424-seg02.mp4"
    REF_VIDEO_PATH  = r"C:\Users\abdul\OneDrive\Desktop\AI BOXING ASSISTANT\TestVideos\NO_HIP_SHOULDER_MOVEMENT-00.00.57.125-00.00.58.185-seg1.mp4"
    PUNCH_TYPE      = "jab"  # jab | cross | lead_hook | rear_hook | lead_uppercut | rear_uppercut
    OUTPUT_DIR      = "feedback_output"
    # ──────────────────────────────────────────────────────────────

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log_path  = os.path.join(OUTPUT_DIR, "feedback_log.txt")
    vid_path  = os.path.join(OUTPUT_DIR, "annotated_punch.mp4")
    log_lines = []

    print("\n[1/4] Extracting user sequence...")
    user_seq, user_lm, user_fps, user_size, user_frames, user_ankle = \
        extract_video_sequence(USER_VIDEO_PATH)
    print(f"      {len(user_seq)} frames  |  {user_size[0]}x{user_size[1]}  |  {user_fps:.0f}fps")

    print("[2/4] Extracting reference sequence...")
    ref_seq, ref_lm, ref_fps, ref_size, _, ref_ankle = \
        extract_video_sequence(REF_VIDEO_PATH)
    print(f"      {len(ref_seq)} frames  |  {ref_size[0]}x{ref_size[1]}  |  {ref_fps:.0f}fps")

    if len(user_seq) == 0 or len(ref_seq) == 0:
        sys.exit("[ERROR] One or both sequences are empty.")

    w, h = user_size

    print("[3/4] Running DTW alignment...")
    u_arr = np.array(user_seq, dtype=np.double)
    r_arr = np.array(ref_seq,  dtype=np.double)
    path  = dtw_ndim.warping_path(u_arr, r_arr, window=20)
    print(f"      DTW distance: {dtw_ndim.distance(u_arr, r_arr, window=20):.2f}")

    print("[4/4] Computing feedback metrics...")
    feedback_list = compute_feedback(
        user_seq, ref_seq, path, PUNCH_TYPE,
        user_ankle, ref_ankle, log_lines
    )

    with open(log_path, "w", encoding="utf-8") as f:
        f.write("\n".join(log_lines))
    print(f"\n      Log saved -> {log_path}")

    print("\nRendering output video...")
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    vw     = cv2.VideoWriter(vid_path, fourcc, user_fps, (w, h))

    path_dict = {}
    for (u_idx, r_idx) in path:
        if u_idx not in path_dict:
            path_dict[u_idx] = r_idx

    for i, frame in enumerate(user_frames):
        fc = frame.copy()
        ref_frame_idx = path_dict.get(i)
        if ref_frame_idx is not None and ref_frame_idx < len(ref_lm) and i < len(user_lm):
            projected_pts = project_reference_onto_user(
                ref_lm[ref_frame_idx], ref_size,
                user_lm[i], user_size,
            )
            draw_reference_skeleton(fc, projected_pts, alpha=0.82)
        cv2.putText(fc, "REFERENCE FORM",
                    (15, h - 18), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 255, 220), 2)
        cv2.putText(fc, f"{i+1}/{len(user_frames)}",
                    (w - 110, 30), cv2.FONT_HERSHEY_SIMPLEX,
                    0.55, (200, 200, 200), 1)
        vw.write(fc)

    last_frame  = user_frames[-1].copy() if user_frames else np.zeros((h, w, 3), np.uint8)
    render_feedback_frames(last_frame, feedback_list, user_fps, vw)

    vw.release()
    print(f"      Video saved -> {vid_path}")

    print("\n--- FEEDBACK SUMMARY ---")
    for fb in feedback_list:
        print(f"  >> {fb}")
    print("------------------------\n")


if __name__ == "__main__":
    main()